# Implementações imediatas:
- [x] Aumentar a captação de população para 1000 metros
- [x] Testar algoritmo exato (p-median) e testar em uma instância pequena para validar a qualidade do resultado exato.
- [ ] Criar tabelas para avaliar melhor os resultados e criar uma tabela comparativa com outros trabalhos


# Realocação de estações pluviométricas
## Download libs

In [1]:
#!pip install pykrige
#!pip install rasterio

# Imports

In [52]:
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pykrige.ok import OrdinaryKriging
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import rasterio
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from rasterio.mask import mask
import folium
from shapely.validation import make_valid
from folium import plugins
from folium.plugins import HeatMap
import pulp
import time


# Leitura dos dados

## Chuva
Dados baixados de plugfield, referentes ao ano de 2025

In [3]:
df_loc = pd.read_excel('Dados/localizacoes.xlsx')

lista_meses = []

for i in range(1, 13):
    nome_arquivo = f"{i:02d}.xlsx"
    caminho_arquivo = "Dados/"+nome_arquivo

    if os.path.exists(caminho_arquivo):
        df_mes = pd.read_excel(caminho_arquivo, skiprows=1)

        df_mes['mes'] = i

        df_mes[['numero', 'regiao', 'nome']] = df_mes['Estação'].str.split(' - ', n=2, expand=True)

        df_mes['numero'] = df_mes['numero'].str.strip().astype(int)
        df_mes['regiao'] = df_mes['regiao'].str.strip()
        df_mes['nome'] = df_mes['nome'].str.strip()

        lista_meses.append(df_mes[['numero', 'Total', 'mes']])

df_chuvas = pd.concat(lista_meses, ignore_index=True)

df_sjc = pd.merge(
    df_chuvas,
    df_loc[['numero', 'nome', 'regiao', 'latitude', 'longitude']],
    on='numero',
    how='left'
)

df_sjc = df_sjc[['numero', 'nome', 'regiao', 'latitude', 'longitude', 'Total', 'mes']]

### Tratamento
Com base em análise prévia, foram identificados algumas estações pluviométricas com comportamento suspeito. As mesmas foram removidas para não influenciar na estimativa de chuva para os testes.

In [4]:
# Remover outlier:
df_sjc['nome'] = df_sjc['nome'].str[1:]

df_sjc = df_sjc[df_sjc['nome'] != "Sítio Bom Jesus"]
df_sjc = df_sjc[df_sjc['nome'] != "Chácaras Miranda"]
df_anual = df_sjc.groupby(['numero', 'nome', 'regiao', 'latitude', 'longitude'])['Total'].sum().reset_index()
df_anual.rename(columns={'Total': 'Acumulado_Anual'}, inplace=True)

In [5]:
df_anual

,numero,nome,regiao,latitude,longitude,Acumulado_Anual
0,1,Rio Comprido,Sul,-23.25859,-45.94240,996.3
1,2,Capuava,Sul,-23.28499,-45.83325,956.1
2,3,Vila Nair,Sul,-23.21662,-45.88589,948.6
3,4,São Bento,Sul,-23.22675,-45.87754,834.9
4,6,Jardim Oriente,Sul,-23.23828,-45.89977,1019.2
5,8,Jardim do Lago,Sudeste,-23.24058,-45.84127,815.7
6,9,Recanto dos Tamoios,Sudeste,-23.27297,-45.79862,866.1
7,10,Vila Corintinha,Leste,-23.18741,-45.86531,961.5
8,11,Jardim Nova América,Centro,-23.19280,-45.90128,725.7
9,12,Jardim das Indústrias,Oeste,-23.22037,-45.92163,1026.0


## Áreas com Risco de Hidrológico
A delimitação das áreas de risco foi extraída da plataforma GeoSanja, o Sistema de Informações Geográficas da Prefeitura de São José dos Campos. Foram utilizados os polígonos correspondentes às áreas de inundação e riscos geológicos mapeados pela Defesa Civil e órgãos municipais. Para garantir a abrangência do modelo e considerar a incerteza espacial, foi aplicado um buffer métrico de 800 metros sobre os polígonos originais, permitindo que o algoritmo de otimização priorize não apenas o ponto exato do risco, mas também o seu entorno imediato.

In [6]:
risco_gdf = gpd.read_file('Dados/pddmap_so.shp').to_crs(epsg=4326)

In [7]:
risco_gdf

,id,name,geometry
0,1,Bacia do Comprido,"POLYGON Z ((-45.88917 -23.29663 0, -45.88903 -..."
1,2,Bacia da Ressaca,"POLYGON Z ((-45.91737 -23.25706 0, -45.9173 -2..."
2,3,Bacia do Vidóca,"POLYGON Z ((-45.86458 -23.28403 0, -45.86452 -..."
3,4,Bacia do Lavapés,"POLYGON Z ((-45.8839 -23.1948 0, -45.88457 -23..."
4,5,Bacia do Putins,"POLYGON Z ((-45.81294 -23.26375 0, -45.81292 -..."
5,6,Bacia do Alambari,"POLYGON Z ((-45.83118 -23.16716 0, -45.83116 -..."
6,7,Bacia dos Veados,"POLYGON Z ((-45.82162 -23.17062 0, -45.82162 -..."
7,8,Bacia do Pararangaba,"POLYGON Z ((-45.8005 -23.19039 0, -45.80045 -2..."
8,9,Bacia do Pararangaba,"POLYGON Z ((-45.79091 -23.14028 0, -45.79088 -..."
9,10,Bacia do Pararangaba,"POLYGON Z ((-45.77868 -23.13484 0, -45.77863 -..."


## Densidade Populacional
Os dados demográficos foram obtidos através do projeto WorldPop, utilizando a estimativa para o ano de 2025. O conjunto de dados fornece a contagem de pessoas por pixel (densidade populacional) com uma resolução espacial de aproximadamente 100 metros (3 arc-seconds). Esta fonte integra dados censitários com técnicas de sensoriamento remoto (luzes noturnas e infraestrutura mapeada) para realizar a desagregação espacial da população, permitindo identificar as áreas de maior vulnerabilidade humana em São José dos Campos.

In [7]:
def get_pop_data(lat_grid, lon_grid, path_worldpop):
    pop_values = []
    with rasterio.open(path_worldpop) as src:
        for lat, lon in zip(lat_grid, lon_grid):
            # Transforma coordenada geográfica em índice de pixel
            row, col = src.index(lon, lat)
            # Lê o valor do pixel (população estimada)
            val = src.read(1)[row, col]
            pop_values.append(val if val > 0 else 0)
    return np.array(pop_values)

In [19]:
#pós correção
def get_pop_data_acumulada(latitudes, longitudes, tiff_path, raio_metros=1000):
    pop_acumulada = []
    
    with rasterio.open(tiff_path) as src:
        # Criar GeoDataFrame dos pontos para converter para UTM e fazer o buffer
        df_temp = pd.DataFrame({'lat': latitudes, 'lon': longitudes})
        gdf_pontos = gpd.GeoDataFrame(
            df_temp, 
            geometry=gpd.points_from_xy(df_temp.lon, df_temp.lat),
            crs="EPSG:4326"
        ).to_crs(epsg=31983) # UTM 23S (São José dos Campos)
        
        gdf_pontos['geometry'] = gdf_pontos.buffer(raio_metros)
        
        gdf_pontos = gdf_pontos.to_crs(src.crs)
        
        for i, row in gdf_pontos.iterrows():
            try:
                out_image, out_transform = mask(src, [row.geometry], crop=True)
                data = out_image[0]
                soma_pixel = data[data > 0].sum()
                pop_acumulada.append(soma_pixel)
            except ValueError:
                pop_acumulada.append(0)
                
    return pop_acumulada

# Interpolação por Krigagem
Referência Clássica (Teoria):

MATHERON, G. (1963). Principles of geostatistics. Economic Geology, v. 58, n. 8, p. 1246-1266.

Referência Aplicada a Chuvas (WMO):

WORLD METEOROLOGICAL ORGANIZATION (WMO). Guide to Hydrological Practices: Volume I: Hydrology - From Measurement to Hydrological Information. WMO-No. 168. 6th ed. Geneva, 2008.

Referência de Implementação (Sistemas de Informação Geográfica):

ISAAKS, E. H.; SRIVASTAVA, R. M. An Introduction to Applied Geostatistics. Oxford University Press, 1989.

## Preparação das Variaveis

In [9]:
x = df_anual['longitude'].values
y = df_anual['latitude'].values
z = df_anual['Acumulado_Anual'].values

## Definição do Modelo
Todos os parâmetros foram definidos com base em busca exaustiva e a seguinte configuração foi a que retornou os melhores resultados com base em validação cruzada:
- modelo_vencedor = 'exponential'
- lags_vencedor = 4

In [10]:
OK = OrdinaryKriging(
    x, y, z,
    variogram_model = 'exponential',
    nlags = 4,
    weight = True
)

# Criação de Base para as análises
A base será criada unindo as informações de Chuva, Risco e Densidade Populacional.

## Criação do grid

In [20]:
lat_min, lat_max = df_anual['latitude'].min(), df_anual['latitude'].max()
lon_min, lon_max = df_anual['longitude'].min(), df_anual['longitude'].max()

lat_margin = (lat_max - lat_min) * 0.05
lon_margin = (lon_max - lon_min) * 0.05

lat_min_exp, lat_max_exp = lat_min - lat_margin, lat_max + lat_margin
lon_min_exp, lon_max_exp = lon_min - lon_margin, lon_max + lon_margin

lats = np.linspace(lat_min_exp, lat_max_exp, 100) 
lons = np.linspace(lon_min_exp, lon_max_exp, 100)
lon_mg, lat_mg = np.meshgrid(lons, lats)

df_grid = pd.DataFrame({
    'latitude': lat_mg.flatten(),
    'longitude': lon_mg.flatten()
})

print(f"Sucesso! Grid gerado com {len(df_grid)} pontos.")
print(f"Limites Lat: {lat_min_exp:.4f} a {lat_max_exp:.4f}")
print(f"Limites Lon: {lon_min_exp:.4f} a {lon_max_exp:.4f}")

Sucesso! Grid gerado com 10000 pontos.
Limites Lat: -23.2973 a -23.0260
Limites Lon: -46.0125 a -45.7466


## Atribuir População:

In [22]:
## Função melhor.
df_grid['populacao'] = get_pop_data_acumulada(
    df_grid['latitude'],
    df_grid['longitude'], 
    'Dados/bra_pop_2025_CN_100m_R2025A_v1.tif'
)

#Aumentar a resolução para 1000 metros (feito)

## Atribuir Risco

In [23]:
#Usar para expandir a área de cada ponto de risco, estou usando 100 metros
pontos_geometria = [Point(xy) for xy in zip(df_grid['longitude'], df_grid['latitude'])]
grid_gdf = gpd.GeoDataFrame(df_grid, geometry=pontos_geometria, crs="EPSG:4326")

risco_utm = risco_gdf.to_crs(epsg=31983)
risco_utm['geometry'] = risco_utm.buffer(100)

risco_buffer = risco_utm.to_crs(epsg=4326)

df_grid['risco'] = grid_gdf.geometry.apply(lambda p: 1 if risco_buffer.contains(p).any() else 0)
print("Distribuição de Risco no Grid:")
print(df_grid['risco'].value_counts())

Distribuição de Risco no Grid:
risco
0    9217
1     783
Name: count, dtype: int64


In [24]:
#Usar essa função se quiser usar apenas os pontos exatos de risco, onde há incidência com algum dos 400 pontos do grid
risco_original_4326 = risco_gdf.to_crs(epsg=4326)

pontos_no_risco = gpd.sjoin(grid_gdf, risco_original_4326, how='left', predicate='intersects')

df_grid['risco'] = pontos_no_risco['index_right'].apply(lambda x: 1 if pd.notnull(x) else 0)

print("Distribuição de Risco no Grid:")
print(df_grid['risco'].value_counts())

Distribuição de Risco no Grid:
risco
0    9799
1     201
Name: count, dtype: int64


## Atribuir Chuva

In [25]:
z_pred, ss_pred = OK.execute('points', df_grid['longitude'].values, df_grid['latitude'].values)

# Adicionar ao DataFrame
df_grid['chuva'] = z_pred


## Normalização dos dados de chuva e população

In [26]:
df_grid['pop_n'] = (df_grid['populacao'] - df_grid['populacao'].min()) / (df_grid['populacao'].max() - df_grid['populacao'].min())
df_grid['chuva_n'] = (df_grid['chuva'] - df_grid['chuva'].min()) / (df_grid['chuva'].max() - df_grid['chuva'].min())

## Finalização

In [27]:
df_grid

,latitude,longitude,populacao,risco,chuva,pop_n,chuva_n
0,-23.297321,-46.012454,1929.499268,0,997.683755,0.055158,0.500652
1,-23.297321,-46.009769,1028.806396,0,997.681782,0.029410,0.500647
2,-23.297321,-46.007084,742.857910,0,997.669331,0.021236,0.500612
3,-23.297321,-46.004398,464.780640,0,997.645893,0.013287,0.500546
4,-23.297321,-46.001713,490.881287,0,997.610944,0.014033,0.500447
...,...,...,...,...,...,...,...
9995,-23.026049,-45.757337,31.253201,0,1012.423289,0.000893,0.542252
9996,-23.026049,-45.754652,34.085667,0,1011.862244,0.000974,0.540669
9997,-23.026049,-45.751966,43.600323,0,1011.303181,0.001246,0.539091
9998,-23.026049,-45.749281,47.344151,0,1010.746817,0.001353,0.537521


In [28]:
print("Grade integrada com sucesso!")
print(f"Total de pontos na grade: {len(df_grid)}")
print(f"Pontos em área de risco (após buffer 800m): {df_grid['risco'].sum()}")
print("-" * 30)
print("Salvar base")
try:
    df_grid.to_parquet('df_grid_processado_novo_final_sjc_100x100.parquet')
    print("Dados salvos em Parquet com sucesso!")
except:
    print("Nota: Para usar Parquet, você pode precisar de: !pip install pyarrow")


Grade integrada com sucesso!
Total de pontos na grade: 10000
Pontos em área de risco (após buffer 800m): 201
------------------------------
Salvar base
Dados salvos em Parquet com sucesso!


# Algoritmos

## K-means
O algoritmo K-means ponderado identifica os centróides de $k$ clusters minimizando a variância espacial das coordenadas do grid, utilizando o peso composto ($W$) para atrair as estações para áreas de maior importância social, meteorológica e de risco. É a abordagem ideal para garantir uma cobertura de área equilibrada e contínua.

In [29]:
def otimizar_kmeans(df_grid, n_p, peso_col='W_exp'):
    """
    Executa o K-means ponderado para encontrar os centroides ideais.
    """
    km = KMeans(n_clusters=n_p, random_state=42, n_init=10)
    km.fit(df_grid[['longitude', 'latitude']], sample_weight=df_grid[peso_col])
    
    df_res = pd.DataFrame(km.cluster_centers_, columns=['longitude', 'latitude'])
    return df_res

## P-Median
O modelo de P-medianas seleciona locais específicos (medianas) que minimizam a distância total ponderada entre os pontos de demanda e as estações. Nesta versão otimizada, o algoritmo seleciona os pontos do grid com os maiores pesos acumulados, garantindo que as estações estejam localizadas exatamente sobre os picos de demanda demográfica e de risco.

### Abordagem Heurística

In [1]:
# Testar o modelo exato com uma instância pequena para alcançar o ótimo e validar a heurística
def otimizar_p_median(df_grid, n_p, peso_col='W_exp', raio_min_km=1.5):
    """
    P-Medians Heurístico: Seleciona pontos de alto peso garantindo 
    uma dispersão mínima para evitar aglomerados.
    """
    raio_graus = raio_min_km / 111.0
    candidatos = df_grid.sort_values(by=peso_col, ascending=False).copy()
    selecionados = []

    for _, ponto in candidatos.iterrows():
        if len(selecionados) >= n_p:
            break
        
        # Verifica se o novo ponto está muito perto dos já selecionados
        ponto_coords = (ponto['latitude'], ponto['longitude'])
        muito_perto = False
        
        for s in selecionados:
            dist = np.sqrt((ponto['latitude'] - s['latitude'])**2 + 
                           (ponto['longitude'] - s['longitude'])**2)
            if dist < raio_graus:
                muito_perto = True
                break
        
        if not muito_perto:
            selecionados.append(ponto)

    return pd.DataFrame(selecionados)[['latitude', 'longitude']].reset_index(drop=True)

## Abordagem Exata - Problema de Cobertura Máxima (MCLP)
Problema clássico onde o objetivo é selecionar um número limitado de conjuntos (ou instalações) para cobrir o maior número possível de elementos (ou pontos de demanda)

In [55]:
def otimizar_cobertura_exata(df_grid_small, n_p, raio_km=0.8, peso_col='W_exp', tempo_max_segundos=600):
    """
    Modelo Exato de Maximização de Cobertura (MCLP).
    """
    coords = df_grid_small[['latitude', 'longitude']].values
    dist_matrix = cdist(coords, coords) * 111.0 # Distância aproximada em km
    pesos = df_grid_small[peso_col].values
    n = len(df_grid_small)
    
    prob = pulp.LpProblem("Max_Cobertura_Exata", pulp.LpMaximize)
    
    # y[j] = 1 se instalamos estação no ponto j
    y = pulp.LpVariable.dicts("y", range(n), cat='Binary')
    # z[i] = 1 se o ponto i está coberto por pelo menos uma estação
    z = pulp.LpVariable.dicts("z", range(n), cat='Binary')
    
    # Maximizar o peso total coberto
    prob += pulp.lpSum(pesos[i] * z[i] for i in range(n))
    
    # Exatamente n_p estações
    prob += pulp.lpSum(y[j] for j in range(n)) == n_p
    
    for i in range(n):
        vizinhos = [j for j in range(n) if dist_matrix[i][j] <= raio_km]
        if vizinhos:
            prob += z[i] <= pulp.lpSum(y[j] for j in vizinhos)
        else:
            prob += z[i] == 0
            
    # Resolve com limite de tempo (ex: 5 minutos) para não travar o PC
    prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=tempo_max_segundos))
    
    estacoes_indices = [j for j in range(n) if pulp.value(y[j]) == 1]
    return df_grid_small.iloc[estacoes_indices][['latitude', 'longitude']].reset_index(drop=True)


## Hybrid (K-means + P-Median, k-means + MCLP))
A abordagem híbrida utiliza o K-means para regionalizar o território em sub-áreas de pesos equivalentes e, em seguida, aplica a lógica de P-medianas dentro de cada cluster. Isso garante que cada região da cidade tenha uma estação localizada no ponto de menor custo de acessibilidade interna, combinando zoneamento espacial com precisão locacional.
Semelhante modo, após a divisão, MCLP exato é executado para otimizar a rede

### Heuristico

In [49]:
def otimizar_hibrido(df_grid, n_p, peso_col='W_exp'):
    """
    Divide o grid em clusters e encontra a mediana local de cada um.
    """
    # 1. Regionalização via K-means
    km = KMeans(n_clusters=n_p, random_state=42, n_init=10)
    clusters = km.fit_predict(df_grid[['longitude', 'latitude']], sample_weight=df_grid[peso_col])
    df_grid['cluster_id'] = clusters
    
    hibrido_coords = []
    
    # 2. Localização da mediana dentro de cada cluster
    for i in range(n_p):
        cluster_data = df_grid[df_grid['cluster_id'] == i]
        if not cluster_data.empty:
            c_coords = cluster_data[['latitude', 'longitude']].values
            c_pesos = cluster_data[peso_col].values
            # Calcula o ponto que minimiza a distância ponderada interna
            d_mat = cdist(c_coords, c_coords)
            idx = np.argmin(np.sum(d_mat * c_pesos, axis=1))
            hibrido_coords.append(cluster_data.iloc[idx][['latitude', 'longitude']])
            
    return pd.DataFrame(hibrido_coords).reset_index(drop=True)

### Exato - MCLP

In [50]:
def otimizar_hibrido_e(df_grid, n_p, peso_col='W_exp', raio_km=0.8):
    """
    1. Divide o grid em 'n_p' clusters via K-Means ponderado.
    2. Em cada cluster, executa o Modelo Exato (MCLP) para encontrar 
       a melhor localização (n_p=1 por cluster).
    """
    from sklearn.cluster import KMeans
    
    # --- Passo 1: Regionalização ---
    # O K-Means define as 'áreas de responsabilidade' de cada estação futura
    km = KMeans(n_clusters=n_p, random_state=42, n_init=10)
    clusters = km.fit_predict(df_grid[['longitude', 'latitude']], sample_weight=df_grid[peso_col])
    df_grid['cluster_id'] = clusters
    
    hibrido_coords = []
    
    # --- Passo 2: Otimização Exata Local ---
    for i in range(n_p):
        cluster_data = df_grid[df_grid['cluster_id'] == i].copy()
        
        if not cluster_data.empty:
            # Executamos o modelo exato para encontrar APENAS 1 estação (n_p=1) 
            # que maximize a cobertura dentro deste cluster específico.
            # O tempo_max_segundos pode ser baixo pois o cluster é pequeno.
            estacao_local = otimizar_cobertura_exata(
                cluster_data, 
                n_p=1, 
                raio_km=raio_km, 
                peso_col=peso_col, 
                tempo_max_segundos=10 
            )
            
            if not estacao_local.empty:
                hibrido_coords.append(estacao_local.iloc[0])
            else:
                # Fallback caso o solver falhe: usa o ponto de maior peso do cluster
                idx_max = cluster_data[peso_col].idxmax()
                hibrido_coords.append(cluster_data.loc[idx_max, ['latitude', 'longitude']])
                
    return pd.DataFrame(hibrido_coords).reset_index(drop=True)

# Testes
A bateria de testes realiza uma análise de sensibilidade multiobjetivo, variando a importância relativa (pesos) entre população, risco e chuva, além de testar multiplicadores de escala para a variável de risco (1, 50 e 100). A avaliação utiliza a métrica de cobertura exclusiva, garantindo que áreas sobrepostas não sejam contabilizadas em duplicidade, permitindo uma comparação justa com a rede vigente de São José dos Campos.

## Testes Auxiliares (P-Means)

In [51]:
# --- 2. NOVA FUNÇÃO DE AVALIAÇÃO (COBERTURA RADIAL) ---
def avaliar_rede_cobertura(coords_result, df_referencia, raio_km=0.8):
    """
    Avalia a rede somando a população e risco de TODOS os pontos do grid
    que estão a uma distância <= raio_km de qualquer estação, sem dupla contagem.
    """
    if coords_result.empty:
        return 0, 0, 0
        
    coords_ref = df_referencia[['latitude', 'longitude']].values
    coords_res = coords_result[['latitude', 'longitude']].values
    
    # Matriz de distância entre o grid inteiro e as estações escolhidas
    dist_matrix = cdist(coords_ref, coords_res) * 111.0
    
    # Distância mínima de cada ponto do grid para a estação mais próxima
    min_dists = dist_matrix.min(axis=1)
    
    # Máscara booleana: Ponto está coberto? (True/False)
    cobertos_mask = min_dists <= raio_km
    
    df_cobertos = df_referencia[cobertos_mask]
    
    pop_total = df_cobertos['populacao'].sum()
    risco_total = df_cobertos['risco'].sum()
    chuva_media = df_cobertos['chuva'].mean() if not df_cobertos.empty else 0
    
    return pop_total, risco_total, chuva_media

# ------------------------------------------------------------------
# EXECUÇÃO DO EXPERIMENTO
# ------------------------------------------------------------------

# Parâmetros
n_p = 31 # Número de estações a instalar
raio_cobertura = 0.8

print(f"Instância de Validação: {len(df_grid_2)} pontos do grid.")

print("Iniciando Modelo Exato (Aguarde, pode demorar)...")
t0_exato = time.time()
res_exato = otimizar_cobertura_exata(df_grid_2, n_p, peso_col='W_exp', raio_km=raio_cobertura)
tempo_exato = time.time() - t0_exato

print("Iniciando Heurística (P-Medians)...")
t0_heur = time.time()
res_heur = otimizar_p_median(df_grid_2, n_p, peso_col='W_exp', raio_min_km=1.5)
tempo_heur = time.time() - t0_heur

# Avaliação Justa (MCLP style)
pop_e, ris_e, chu_e = avaliar_rede_cobertura(res_exato, df_grid_2, raio_km=raio_cobertura)
pop_h, ris_h, chu_h = avaliar_rede_cobertura(res_heur, df_grid_2, raio_km=raio_cobertura)

# Construção da Tabela com os Tempos Exatos
df_comp = pd.DataFrame({
    'Métrica': ['População Coberta', 'Pontos de Risco', 'Chuva Média (mm)'],
    'Modelo Exato (Ótimo)': [f"{pop_e:.1f}", f"{ris_e:.0f}", f"{chu_e:.2f}"],
    'Sua Heurística': [f"{pop_h:.1f}", f"{ris_h:.0f}", f"{chu_h:.2f}"],
    'Eficiência da Heurística': [
        f"{(pop_h/pop_e)*100:.1f}%" if pop_e > 0 else "-", 
        f"{(ris_h/ris_e)*100:.1f}%" if ris_e > 0 else "-", 
        "-"
    ]
})

print("\n========================================================")
print("   RESULTADO DA VALIDAÇÃO: EXATO vs HEURÍSTICO")
print("========================================================")
print(df_comp.to_string(index=False))
print("--------------------------------------------------------")
print(f"Tempo Modelo Exato : {tempo_exato:.2f} segundos")
print(f"Tempo Heurística   : {tempo_heur:.4f} segundos")
print(f"Aceleração (Speedup): {tempo_exato/tempo_heur:.1f}x mais rápida")
print("========================================================")

Instância de Validação: 10000 pontos do grid.
Iniciando Modelo Exato (Aguarde, pode demorar)...
Iniciando Heurística (P-Medians)...

   RESULTADO DA VALIDAÇÃO: EXATO vs HEURÍSTICO
          Métrica Modelo Exato (Ótimo) Sua Heurística Eficiência da Heurística
População Coberta            3759580.2      7705744.5                   205.0%
  Pontos de Risco                  361            194                    53.7%
 Chuva Média (mm)               959.46        1024.33                        -
--------------------------------------------------------
Tempo Modelo Exato : 38.50 segundos
Tempo Heurística   : 0.0384 segundos
Aceleração (Speedup): 1003.3x mais rápida


In [41]:
# Parâmetros
n_p = 31 # Número de estações a instalar

# --- 1. Execução do Modelo EXATO (PuLP) ---
print("Iniciando Modelo Exato...")
t0_exato = time.time()
res_exato = otimizar_cobertura_exata(df_grid_2, n_p, peso_col='W_exp')
tempo_exato = time.time() - t0_exato

# --- 2. Execução da Sua HEURÍSTICA (P-Medians c/ Repulsão) ---
print("Iniciando Heurística...")
t0_heur = time.time()
# Usando o raio_min_km que definimos anteriormente para evitar sobreposição
res_heur = otimizar_p_median(df_grid_2, n_p, peso_col='W_exp', raio_min_km=1.5)
tempo_heur = time.time() - t0_heur

# --- 3. Função de Avaliação de Resultados ---
def avaliar_rede(coords_result, df_referencia):
    # Soma da população, risco e média de chuva nos pontos selecionados
    pop_total = coords_result.merge(df_referencia, on=['latitude', 'longitude'])['populacao'].sum()
    risco_total = coords_result.merge(df_referencia, on=['latitude', 'longitude'])['risco'].sum()
    chuva_media = coords_result.merge(df_referencia, on=['latitude', 'longitude'])['chuva'].mean()
    return pop_total, risco_total, chuva_media

pop_e, ris_e, chu_e = avaliar_rede(res_exato, df_grid_2)
pop_h, ris_h, chu_h = avaliar_rede(res_heur, df_grid_2)

# --- 4. Tabela Comparativa Final ---
df_comp = pd.DataFrame({
    'Métrica': ['População Coberta', 'Pontos de Risco', 'Chuva Média (mm)', 'Tempo de Execução (s)'],
    'Modelo Exato (Ótimo)': [pop_e, ris_e, f"{chu_e:.2f}", f"{tempo_exato:.4f}"],
    'Sua Heurística': [pop_h, ris_h, f"{chu_h:.2f}", f"{tempo_heur:.4f}"],
    'Eficiência (%)': [
        f"{(pop_h/pop_e)*100:.1f}%", 
        f"{(ris_h/ris_e)*100 if ris_e > 0 else 100:.1f}%", 
        "-", 
        f"{(tempo_exato/tempo_heur):.1f}x mais rápida"
    ]
})

print("\n========================================================")
print("   RESULTADO DA VALIDAÇÃO: EXATO vs HEURÍSTICO")
print("========================================================")
print(df_comp.to_string(index=False))

Iniciando Modelo Exato...
Iniciando Heurística...

   RESULTADO DA VALIDAÇÃO: EXATO vs HEURÍSTICO
              Métrica Modelo Exato (Ótimo) Sua Heurística     Eficiência (%)
    População Coberta           186194.375    396000.4375             212.7%
      Pontos de Risco                   16             31             193.8%
     Chuva Média (mm)               959.87        1025.18                  -
Tempo de Execução (s)              37.7676         0.0559 675.2x mais rápida


## Testes Principais

In [57]:
TOTAL_POP_REAL_SJC = 729737  # Censo IBGE aproximado
pop_no_grid_total = df_grid['populacao'].sum()
fator_escala_pop = TOTAL_POP_REAL_SJC / pop_no_grid_total if pop_no_grid_total > 0 else 1

def avaliar_rede_final_desnormalizada(coords_otimizadas, df_grid_original, fator_pop):
    """
    Avalia a rede retornando valores reais aproximados E a pontuação de otimização.
    """
    if coords_otimizadas is None or coords_otimizadas.empty:
        return 0, 0, 0, 0

    # Usando cdist para performance (distância aproximada de 800m = 0.0072 graus)
    coords_g = df_grid_original[['latitude', 'longitude']].values
    coords_e = coords_otimizadas[['latitude', 'longitude']].values
    distancias = cdist(coords_g, coords_e)
    cobertura_total = np.any(distancias <= 0.0072, axis=1)
    
    grid_coberto = df_grid_original[cobertura_total]
    
    # 1. População desnormalizada
    pop_real = grid_coberto['populacao'].sum() * fator_pop
    
    # 2. Risco (Contagem de células)
    risco_pontos = grid_coberto['risco'].sum() 
    
    # 3. Chuva (Média)
    chuva_val = grid_coberto['chuva'].mean() if not grid_coberto.empty else 0
    
    # 4. PONTUAÇÃO (Soma do Peso W_exp que o algoritmo de fato enxergou e otimizou)
    pontuacao = grid_coberto['W_exp'].sum() if 'W_exp' in grid_coberto.columns else 0
    
    return pop_real, risco_pontos, chuva_val, pontuacao

configuracoes = [
    {"nome": "Balanceado (1/3)", "pop": 0.333, "risco": 0.333, "chuva": 0.334},
    {"nome": "Foco População (50%)", "pop": 0.50, "risco": 0.25, "chuva": 0.25},
    {"nome": "Foco Risco (50%)", "pop": 0.25, "risco": 0.50, "chuva": 0.25},
    {"nome": "Foco Chuva (50%)", "pop": 0.25, "risco": 0.25, "chuva": 0.50}
]
fatores_risco = [1, 5, 10, 50, 100]

n_estacoes = len(x) # Considerando que 'x' é a lista de longitudes originais
df_rede_atual = pd.DataFrame({'longitude': x, 'latitude': y})

historico_redes = {}
melhores_redes = {
    'populacao': {'valor': -1, 'coords': None, 'config': ""},
    'risco': {'valor': -1, 'coords': None, 'config': ""},
    'chuva': {'valor': -1, 'coords': None, 'config': ""},
    'pontuacao': {'valor': -1, 'coords': None, 'config': ""}
}

for conf in configuracoes:
    for f_risco in fatores_risco:
        # 1. Atualizar a superfície de custo (W_exp) para o cenário atual
        W = (df_grid['pop_n'] * conf['pop']) + \
            ((df_grid['risco'] * f_risco) * conf['risco']) + \
            (df_grid['chuva_n'] * conf['chuva'])
        df_grid['W_exp'] = W

        # 2. Avaliar a Rede Atual com o W_exp deste cenário específico
        stats_real = avaliar_rede_final_desnormalizada(df_rede_atual, df_grid, fator_escala_pop)

        # 3. Execuções e Cronometragem
        t0 = time.time()
        res_km_coords = otimizar_kmeans(df_grid, n_estacoes)
        t_km = time.time() - t0
        
        t0 = time.time()
        res_pm_coords = otimizar_p_median(df_grid, n_estacoes) # Heurística
        t_pm = time.time() - t0
        
        t0 = time.time()
        res_hb_coords = otimizar_hibrido(df_grid, n_estacoes)
        t_hb = time.time() - t0

        t0 = time.time()
        res_ex_coords = otimizar_cobertura_exata(df_grid, n_estacoes, raio_km=0.8, peso_col='W_exp')
        t_ex = time.time() - t0

        t0 = time.time()
        res_hbe_coords = otimizar_hibrido_e(df_grid, n_estacoes)
        t_hbe = time.time() - t0

        # 4. Avaliação dos resultados
        res_km = avaliar_rede_final_desnormalizada(res_km_coords, df_grid, fator_escala_pop)
        res_pm = avaliar_rede_final_desnormalizada(res_pm_coords, df_grid, fator_escala_pop)
        res_hb = avaliar_rede_final_desnormalizada(res_hb_coords, df_grid, fator_escala_pop)
        res_ex = avaliar_rede_final_desnormalizada(res_ex_coords, df_grid, fator_escala_pop)
        res_hbe = avaliar_rede_final_desnormalizada(res_hbe_coords, df_grid, fator_escala_pop)
        
        # Armazenar histórico
        rodada_info = {
            'K-Means': {'coords': res_km_coords, 'stats': res_km, 'tempo': t_km},
            'Heurística PM': {'coords': res_pm_coords, 'stats': res_pm, 'tempo': t_pm},
            'Híbrido': {'coords': res_hb_coords, 'stats': res_hb, 'tempo': t_hb},
            'Exato (MCLP)': {'coords': res_ex_coords, 'stats': res_ex, 'tempo': t_ex},
            'Híbrido_e': {'coords': res_hbe_coords, 'stats': res_hbe, 'tempo': t_hbe},
        }
        
        for metodo, info in rodada_info.items():
            chave = f"{conf['nome']}_F{f_risco}_{metodo}"
            historico_redes[chave] = info
            
            p, r, c, pont = info['stats']
            if p > melhores_redes['populacao']['valor']:
                melhores_redes['populacao'] = {'valor': p, 'coords': info['coords'], 'config': chave}
            if r > melhores_redes['risco']['valor']:
                melhores_redes['risco'] = {'valor': r, 'coords': info['coords'], 'config': chave}
            if c > melhores_redes['chuva']['valor']:
                melhores_redes['chuva'] = {'valor': c, 'coords': info['coords'], 'config': chave}
            if pont > melhores_redes['pontuacao']['valor']:
                melhores_redes['pontuacao'] = {'valor': pont, 'coords': info['coords'], 'config': chave}

        # 5. Tabela de Apresentação
        # Função para evitar divisão por zero no ganho
        def calc_ganho(novo, antigo):
            return ((novo / antigo) - 1) * 100 if antigo > 0 else 0

        data = [
            ["Rede Vigente", stats_real[0], 0, stats_real[1], 0, stats_real[2], 0, stats_real[3], 0, 0.0],
            ["K-Means", res_km[0], calc_ganho(res_km[0], stats_real[0]), 
                        res_km[1], calc_ganho(res_km[1], stats_real[1]), 
                        res_km[2], calc_ganho(res_km[2], stats_real[2]),
                        res_km[3], calc_ganho(res_km[3], stats_real[3]), t_km],
            ["Heurística PM", res_pm[0], calc_ganho(res_pm[0], stats_real[0]), 
                              res_pm[1], calc_ganho(res_pm[1], stats_real[1]), 
                              res_pm[2], calc_ganho(res_pm[2], stats_real[2]),
                              res_pm[3], calc_ganho(res_pm[3], stats_real[3]), t_pm],
            ["Híbrido", res_hb[0], calc_ganho(res_hb[0], stats_real[0]), 
                        res_hb[1], calc_ganho(res_hb[1], stats_real[1]), 
                        res_hb[2], calc_ganho(res_hb[2], stats_real[2]),
                        res_hb[3], calc_ganho(res_hb[3], stats_real[3]), t_hb],
            ["Exato (MCLP)", res_ex[0], calc_ganho(res_ex[0], stats_real[0]), 
                             res_ex[1], calc_ganho(res_ex[1], stats_real[1]), 
                             res_ex[2], calc_ganho(res_ex[2], stats_real[2]),
                             res_ex[3], calc_ganho(res_ex[3], stats_real[3]), t_ex],
            ["Híbrido_E", res_hbe[0], calc_ganho(res_hbe[0], stats_real[0]), 
                        res_hbe[1], calc_ganho(res_hbe[1], stats_real[1]), 
                        res_hbe[2], calc_ganho(res_hbe[2], stats_real[2]),
                        res_hbe[3], calc_ganho(res_hbe[3], stats_real[3]), t_hbe],
        ]

        cols = ["Método", "População", "Ganho Pop%", "Risco", "Ganho Risco%", "Chuva", "Ganho Chuva%", "Pontuação", "Ganho Pts%", "Tempo (s)"]
        df_res = pd.DataFrame(data, columns=cols)
        
        print(f"\n" + "="*130)
        print(f"EXPERIMENTO: {conf['nome']} | FATOR RISCO: {f_risco}")
        print("="*130)
        print(df_res.to_string(index=False, float_format=lambda x: "{:.2f}".format(x)))


EXPERIMENTO: Balanceado (1/3) | FATOR RISCO: 1
       Método  População  Ganho Pop%  Risco  Ganho Risco%   Chuva  Ganho Chuva%  Pontuação  Ganho Pts%  Tempo (s)
 Rede Vigente   97392.88        0.00     23          0.00  994.05          0.00     159.03        0.00       0.00
      K-Means   78212.83      -19.69     18        -21.74 1007.96          1.40     159.82        0.50       0.28
Heurística PM  155697.31       59.87     75        226.09 1004.83          1.08     201.63       26.79       0.02
      Híbrido   75073.69      -22.92     19        -17.39 1008.96          1.50     152.93       -3.84       0.32
 Exato (MCLP)  245420.88      151.99     38         65.22 1023.70          2.98     238.26       49.82     428.59
    Híbrido_E  108191.45       11.09     43         86.96 1025.54          3.17     184.70       16.14       5.68

EXPERIMENTO: Balanceado (1/3) | FATOR RISCO: 5
       Método  População  Ganho Pop%  Risco  Ganho Risco%   Chuva  Ganho Chuva%  Pontuação  Ganho Pts%  Te

# Resultados visuais

In [ ]:
# 1. O que mais protegeu a população (P-Medians | Foco População | F1) (DESATUALIZADO)
cenario_pop = melhores_redes['populacao']

# 2. O que mais cobriu áreas de risco (Híbrido | Balanceado | F100) (DESATUALIZADO)
cenario_risco = melhores_redes['risco']

## População

In [69]:
cenario_p = melhores_redes['populacao']
mapa_pop = folium.Map(location=[-23.18, -45.88], zoom_start=12, tiles='cartodbpositron')

heat_data = [[row['latitude'], row['longitude'], row['populacao']] for _, row in df_grid.iterrows() if row['populacao'] > 0]
HeatMap(heat_data, radius=15, blur=20, name="Densidade Populacional").add_to(mapa_pop)

folium.GeoJson(risco_gdf.to_crs(epsg=4326).geometry, name="Zonas de Risco",
               style_function=lambda x: {'color': '#7f8c8d', 'weight': 1, 'fillOpacity': 0.1}).add_to(mapa_pop)

group_ref = folium.FeatureGroup(name="Rede Vigente (Estado Atual)").add_to(mapa_pop)
for _, row in df_rede_atual.iterrows():
    folium.Circle(location=[row['latitude'], row['longitude']], radius=800, 
                  color='#7f8c8d', weight=1.5, fill=True, fill_opacity=0.1).add_to(group_ref)
    folium.CircleMarker(location=[row['latitude'], row['longitude']], radius=2, color='#34495e', fill=True).add_to(group_ref)

group_otim_p = folium.FeatureGroup(name="Rede Proposta (Máxima População)").add_to(mapa_pop)
for _, row in cenario_p['coords'].iterrows():
    folium.Circle(location=[float(row['latitude']), float(row['longitude'])], radius=800, 
                  color='blue', weight=2, fill=True, fill_opacity=0.2).add_to(group_otim_p)
    folium.Marker(location=[float(row['latitude']), float(row['longitude'])],
                  icon=folium.Icon(color='blue', icon='users', prefix='fa')).add_to(group_otim_p)

Fullscreen().add_to(mapa_pop)
folium.LayerControl(collapsed=False).add_to(mapa_pop)
mapa_pop.save("Mapa_PhD_Pop_v_Vigente.html")

## Risco

In [ ]:
cenario_r = melhores_redes['risco']
mapa_risco = folium.Map(location=[-23.18, -45.88], zoom_start=12, tiles='cartodbpositron')

folium.GeoJson(risco_gdf.to_crs(epsg=4326).geometry, name="Risco Crítico",
               style_function=lambda x: {'fillColor': '#e74c3c', 'color': '#c0392b', 'weight': 2, 'fillOpacity': 0.5}).add_to(mapa_risco)

group_ref_r = folium.FeatureGroup(name="Rede Vigente (Estado Atual)").add_to(mapa_risco)
for _, row in df_rede_atual.iterrows():
    folium.Circle(location=[row['latitude'], row['longitude']], radius=800, 
                  color='#95a5a6', weight=1, dash_array='5, 5', fill=False).add_to(group_ref_r)
    folium.CircleMarker(location=[row['latitude'], row['longitude']], radius=2, color='#7f8c8d', fill=True).add_to(group_ref_r)

group_otim_r = folium.FeatureGroup(name="Rede Proposta (Máximo Risco)").add_to(mapa_risco)
for _, row in cenario_r['coords'].iterrows():
    folium.Circle(location=[float(row['latitude']), float(row['longitude'])], radius=800, 
                  color='darkred', weight=2, fill=True, fill_opacity=0.2).add_to(group_otim_r)
    folium.Marker(location=[float(row['latitude']), float(row['longitude'])],
                  icon=folium.Icon(color='darkred', icon='exclamation-triangle', prefix='fa')).add_to(group_otim_r)

Fullscreen().add_to(mapa_risco)
folium.LayerControl(collapsed=False).add_to(mapa_risco)
mapa_risco.save("Mapa_PhD_Risco_v_Vigente.html")